# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template and example for loading and exploring the FAIR² Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Name:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("License:", metadata.license)
print("Keywords:", metadata.keywords)
print("Personal Sensitive Information:", metadata.personalSensitiveInformation)

## 2. Data Overview
Review available record sets, fields, and their `@id`s.
All dataset entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# List available record sets by @id
record_sets = dataset.metadata.recordSet
if not record_sets:
    print("No record sets found in dataset metadata.")
else:
    for rs in record_sets:
        rs_id = rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs
        print(f"RecordSet @id: {rs_id}")

In [ ]:
# If dataset.metadata.recordSet is empty, retrieve record set ids dynamically
# mlcroissant exposes .record_sets if present, else records can be loaded directly
try:
    available_record_sets = dataset.record_sets()
except AttributeError:
    # fallback if record_sets() not implemented
    available_record_sets = []

if available_record_sets:
    print("Available RecordSets:")
    for rs in available_record_sets:
        print(rs)
else:
    print("Fetching records...")
    # Try loading with a known record set id, otherwise show example,
    # For illustrative purposes, provide the record set id (replace with actual id):
    example_record_set_id = 'https://api.app.sen.science/frontiers/7160411/mental-health-survey-records-main'  # Replace with actual if known
    try:
        for x in dataset.records(record_set=example_record_set_id):
            print(x)
    except Exception as e:
        print("Example record retrieval did not work. Please check dataset schema for record set IDs.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use record set and field `@id`s from the overview section above.

For this dataset, use the primary record set id (replace with actual `@id` if known).

In [ ]:
# Replace with actual record set IDs from the dataset
record_set_ids = [
    'https://api.app.sen.science/frontiers/7160411/mental-health-survey-records-main'
    # Add other @id if there are multiple record sets
]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"DataFrame columns for RecordSet {record_set_id}:")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not retrieve records for RecordSet {record_set_id}: {str(e)}")

# For further EDA, pick the first record set
primary_rs_id = record_set_ids[0]
df = dataframes.get(primary_rs_id, pd.DataFrame())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalization, grouping.
All fields referenced by their `@id`.

Example: Filter and normalize GAD-7 scores, group by gender.

In [ ]:
# Example numeric field and group field @ids
numeric_field_id = 'https://api.app.sen.science/frontiers/7160411/gad7_score'  # GAD-7 score field
group_field_id = 'https://api.app.sen.science/frontiers/7160411/gender'       # Gender field

if df.empty:
    print("DataFrame is empty. Please check data extraction.")
else:
    col_ids = df.columns.tolist()
    print("Available columns (@id):", col_ids)

    # Check if numeric field and group field exist
    if numeric_field_id in col_ids:
        threshold = 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by gender field
        if group_field_id in col_ids:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (average {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print(f"Field {numeric_field_id} not found. Please check available columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Example: Histogram of GAD-7 scores and bar plot of average GAD-7 by gender (fields referenced by `@id`).

In [ ]:
if not df.empty and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    plt.hist(df[numeric_field_id].dropna(), bins=10, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of GAD-7 Scores ({numeric_field_id})")
    plt.xlabel("GAD-7 Score")
    plt.ylabel("Count")
    plt.show()

    if group_field_id in df.columns:
        group_means = df.groupby(group_field_id)[numeric_field_id].mean()
        group_means.plot(kind='bar', color='orange', edgecolor='black')
        plt.title(f"Average GAD-7 Score by Gender ({group_field_id})")
        plt.xlabel("Gender")
        plt.ylabel("Average GAD-7 Score")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR² Mental Health Survey dataset from Kilifi County, Kenya using `mlcroissant`.
- We demonstrated dataset extraction by referencing all entities via their `@id` fields.
- Basic EDA and visualizations showed the distribution and grouping of GAD-7 anxiety scores by gender.
- The same approach can be applied for other fields (e.g., PHQ-9, PSQ) and to deeper demographic or survey-based analyses.

**Note:** All field and record set references should use the exact `@id` as specified in the Croissant schema for reproducibility and proper metadata linkage.